<a href="https://colab.research.google.com/github/ChronosXV/Metodologia_grupo2/blob/noelia_modelado/Metodologia_1_Grupo_2_Modelado_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Trabajo Práctico Final

**Materia:**

Metodología de la Investigación

**Alumnos:**

Gisela Martínez
Noelia Cualina
José Leonardo Carabajal
Javier Carabajal


**Objetivo:**

Desarrollar y evaluar modelos de Machine Learning para predecir niveles de demanda en la red de subtes de CABA.

Período de estudio: 2017–2021.



Enfoque: cuantitativo, descriptivo y predictivo.



Metodología: CRISP-DM.



Variable objetivo: nivel_demanda, con categorías baja, media y alta.



Variable base para construir el objetivo: pax_total.



Variables predictoras: línea, estación, franja horaria, año, mes, día de la semana y condición de día hábil.



Evaluación: accuracy, precision, recall, F1-score y matriz de confusión.

###1. Introducción y objetivo del modelado

El presente trabajo constituye la continuación del análisis exploratorio y la preparación de datos realizados sobre los registros históricos de pasajeros de la red de subterráneos de la Ciudad Autónoma de Buenos Aires durante el período 2017–2021.

En esta etapa se propone desarrollar y evaluar modelos de Machine Learning que permitan clasificar y predecir niveles de demanda de pasajeros, utilizando información temporal y espacial disponible en los registros históricos.

Para ello, se utilizará el conjunto de datos previamente procesado y agrupado por fecha, línea, estación y franja horaria, tomando como referencia la cantidad total de pasajeros registrados (pax_total).

A partir de esta variable se construirán tres categorías de demanda: baja, media y alta, que constituirán la variable objetivo (nivel_demanda) del problema de clasificación supervisada.

Los modelos utilizarán características como la línea, la estación, la franja horaria y variables temporales derivadas de la fecha. La cantidad total de pasajeros no será incorporada como variable predictora, dado que se utilizará para construir la variable objetivo.

Se compararán distintos algoritmos de clasificación mediante métricas de evaluación apropiadas, prestando especial atención a la identificación de situaciones de alta demanda.

Asimismo, se contempla la aplicación de técnicas de aprendizaje no supervisado como análisis complementario, con el propósito de identificar posibles agrupamientos en los patrones de demanda.

El desarrollo seguirá las etapas de modelado y evaluación de la metodología CRISP-DM, buscando obtener resultados reproducibles que permitan contrastar la hipótesis planteada y formular recomendaciones basadas en evidencia cuantitativa.
####

**Hipótesis de investigación**

Los episodios de alta demanda en la red de subtes de CABA durante el período 2017–2021 no se distribuyen de manera uniforme, sino que presentan patrones asociados a factores temporales y espaciales. Se espera que la probabilidad de registrar niveles elevados de demanda varíe según la estación, la línea, el día de la semana y la franja horaria, permitiendo identificar y predecir situaciones de alta demanda a partir de estas características.


###2. Importación de librerías y carga del dataset procesado.

En esta sección se importan las bibliotecas necesarias para el procesamiento de datos, la construcción de modelos de Machine Learning y la evaluación de resultados. Posteriormente, se carga el conjunto de datos consolidado durante la etapa de preparación, que servirá como base para el modelado predictivo.

In [1]:
# Manipulación y análisis de datos
import pandas as pd
import numpy as np
import os

# Visualización de resultados
import matplotlib.pyplot as plt
import seaborn as sns

# Preparación de datos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos de clasificación supervisada
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluación de modelos
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

# Aprendizaje no supervisado
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Acceso a Google Drive
from google.colab import drive

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [2]:
# Montar Google Drive
drive.mount("/content/drive")

# Definir la ruta del dataset procesado
ruta_carpeta = "/content/drive/MyDrive/Datos_csv/"
ruta_limpios = os.path.join(ruta_carpeta, "datos_limpios")

ruta_dataset = os.path.join(
    ruta_limpios,
    "historico_2017_2021_dim.parquet"
)

# Cargar el dataset consolidado
df = pd.read_parquet(ruta_dataset)

# Verificar la carga
print("Dataset cargado correctamente.")
print(f"Cantidad de registros: {df.shape[0]:,}")
print(f"Cantidad de columnas: {df.shape[1]}")

Mounted at /content/drive
Dataset cargado correctamente.
Cantidad de registros: 653,619
Cantidad de columnas: 6


In [3]:
display(df.head())

,fecha,linea,estacion,franja_horaria_num,franja_horaria,pax_total
0,2017-11-27,D,9 DE JULIO,5,Noche,535.0
1,2017-11-28,D,9 DE JULIO,1,Madrugada,9.0
2,2017-11-28,D,9 DE JULIO,2,Pico mañana,1922.0
3,2017-11-28,D,9 DE JULIO,3,Media mañana,4345.0
4,2017-11-29,D,9 DE JULIO,2,Pico mañana,1722.0


In [4]:
# Verificar estructura y tipos de datos del dataset
print("Información general del dataset:")
df.info()

print("\nValores nulos por columna:")
print(df.isnull().sum())

Información general del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 653619 entries, 0 to 653618
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   fecha               653619 non-null  object 
 1   linea               653619 non-null  object 
 2   estacion            653619 non-null  object 
 3   franja_horaria_num  653619 non-null  int32  
 4   franja_horaria      653619 non-null  object 
 5   pax_total           653619 non-null  float64
dtypes: float64(1), int32(1), object(4)
memory usage: 27.4+ MB

Valores nulos por columna:
fecha                 0
linea                 0
estacion              0
franja_horaria_num    0
franja_horaria        0
pax_total             0
dtype: int64


###3. Construcción de variables predictoras.

En esta etapa se preparan las variables que serán utilizadas como características de entrada para los modelos de Machine Learning.

A partir del conjunto de datos consolidado, se conservarán las variables espaciales y temporales disponibles, como la línea, la estación y la franja horaria. Además, se construirán nuevas variables a partir de la fecha, con el propósito de representar patrones asociados al comportamiento de la demanda de pasajeros.

Las variables derivadas serán:

*  Año: permite identificar variaciones de demanda entre los distintos años del período analizado.
*  Mes: permite representar posibles patrones estacionales.
*  Día de la semana: permite identificar diferencias de demanda entre lunes y domingo.
*  Fin de semana: permite distinguir sábados y domingos del resto de los días.


La variable pax_total se conservará únicamente para la posterior construcción de la variable objetivo nivel_demanda, correspondiente a las categorías baja, media y alta.

De esta manera, se busca que los modelos aprendan relaciones entre las características temporales y espaciales y los niveles de demanda, evitando incorporar directamente como predictor la cantidad de pasajeros que se desea clasificar.

In [5]:
# Crear una copia del dataset para el modelado
df_modelo = df.copy()

# Convertir la columna fecha a tipo datetime
df_modelo["fecha"] = pd.to_datetime(
    df_modelo["fecha"],
    format="%Y-%m-%d",
    errors="raise"
)

# Construir variables temporales
df_modelo["anio"] = df_modelo["fecha"].dt.year
df_modelo["mes"] = df_modelo["fecha"].dt.month
df_modelo["dia_semana"] = df_modelo["fecha"].dt.dayofweek

# Identificar fines de semana: sábado (5) y domingo (6)
df_modelo["fin_semana"] = (
    df_modelo["dia_semana"] >= 5
).astype(int)

# Visualizar los primeros registros
display(df_modelo.head())

print("\nDimensiones del dataset:", df_modelo.shape)

,fecha,linea,estacion,franja_horaria_num,franja_horaria,pax_total,anio,mes,dia_semana,fin_semana
0,2017-11-27,D,9 DE JULIO,5,Noche,535.0,2017,11,0,0
1,2017-11-28,D,9 DE JULIO,1,Madrugada,9.0,2017,11,1,0
2,2017-11-28,D,9 DE JULIO,2,Pico mañana,1922.0,2017,11,1,0
3,2017-11-28,D,9 DE JULIO,3,Media mañana,4345.0,2017,11,1,0
4,2017-11-29,D,9 DE JULIO,2,Pico mañana,1722.0,2017,11,2,0



Dimensiones del dataset: (653619, 10)


In [6]:
# Seleccionar las variables predictoras
variables_predictoras = [
    "linea",
    "estacion",
    "franja_horaria_num",
    "anio",
    "mes",
    "dia_semana"
]

# Crear el conjunto de características
X = df_modelo[variables_predictoras].copy()

# Verificar las variables seleccionadas
print("Variables predictoras:")
print(X.columns.tolist())

print("\nDimensiones de X:", X.shape)

display(X.head())

Variables predictoras:
['linea', 'estacion', 'franja_horaria_num', 'anio', 'mes', 'dia_semana']

Dimensiones de X: (653619, 6)


,linea,estacion,franja_horaria_num,anio,mes,dia_semana
0,D,9 DE JULIO,5,2017,11,0
1,D,9 DE JULIO,1,2017,11,1
2,D,9 DE JULIO,2,2017,11,1
3,D,9 DE JULIO,3,2017,11,1
4,D,9 DE JULIO,2,2017,11,2


###4. Definición de la variable objetivo nivel_demanda.

Para el desarrollo de los modelos de clasificación supervisada, se construirá una variable objetivo denominada nivel_demanda, que permitirá clasificar los registros históricos de pasajeros en tres categorías: baja, media y alta demanda.

Esta clasificación se realizará a partir de la variable pax_total, que representa la cantidad total de pasajeros registrada para cada combinación de fecha, línea, estación y franja horaria.

Se propone utilizar un criterio basado en cuantiles, dividiendo la distribución de pasajeros en tres grupos mediante los percentiles 33,33 y 66,67. Este procedimiento permitirá establecer umbrales reproducibles a partir de la distribución observada.

Para evitar fugas de información, los umbrales definitivos se calcularán exclusivamente sobre el conjunto de entrenamiento, después de realizar la separación temporal de los datos. Posteriormente, se aplicarán los mismos límites al conjunto de prueba.

La variable pax_total no se utilizará como característica predictora, ya que constituye la información a partir de la cual se define la variable objetivo.

Es importante señalar que los niveles establecidos representan categorías relativas de demanda de pasajeros, utilizadas como aproximación para identificar situaciones potenciales de congestión, y no mediciones directas de ocupación o saturación de las estaciones.

In [7]:
# Separación temporal preliminar del dataset
# Entrenamiento: 2017 a 2020
# Prueba: 2021

df_train = df_modelo[
    df_modelo["anio"] <= 2020
].copy()

df_test = df_modelo[
    df_modelo["anio"] == 2021
].copy()

# Verificar la cantidad de registros
print("Registros de entrenamiento:", len(df_train))
print("Registros de prueba:", len(df_test))

print("\nDistribución porcentual:")
print(
    f"Entrenamiento: {len(df_train) / len(df_modelo) * 100:.2f}%"
)

Registros de entrenamiento: 541247
Registros de prueba: 112372

Distribución porcentual:
Entrenamiento: 82.81%


In [9]:
# Calcular los cuantiles únicamente con datos de entrenamiento
cuantiles = df_train["pax_total"].quantile([1/3, 2/3])

# Mostrar los valores obtenidos
print("Cuantiles de pax_total (2017-2020):")
print(cuantiles)

Cuantiles de pax_total (2017-2020):
0.333333     354.0
0.666667    1825.0
Name: pax_total, dtype: float64


**Interpretación de los cuantiles obtenidos**

A partir de los registros correspondientes al conjunto de entrenamiento (2017–2020), se calcularon los percentiles 33,33 y 66,67 de la variable `pax_total`, obteniendo los siguientes resultados:

| Percentil | Pasajeros | Interpretación |
|---|---:|---|
| 33,33 % | 354 | Límite entre demanda baja y media |
| 66,67 % | 1.825 | Límite entre demanda media y alta |

Estos valores permiten establecer tres niveles de demanda:

- **Demanda baja:** registros con hasta 354 pasajeros.
- **Demanda media:** registros con más de 354 y hasta 1.825 pasajeros.
- **Demanda alta:** registros con más de 1.825 pasajeros.

Los umbrales se definieron utilizando exclusivamente los datos de entrenamiento y se aplicarán posteriormente al conjunto de prueba (2021), manteniendo el mismo criterio de clasificación.

Es importante destacar que estas categorías representan niveles relativos de demanda según la distribución histórica de pasajeros, y no umbrales operativos de congestión establecidos por la capacidad de las estaciones.

###5. Preparación de datos y separación temporal.

xxxx

###6. Entrenamiento de modelos supervisados.

xxxx

###7. Evaluación y comparación de resultados.

xxxx

###8. Análisis complementario con clustering.

xxxx

###9. Interpretación y conclusiones.